<a href="https://colab.research.google.com/github/hristinagjorgjievska/Natural-Language-Processing/blob/master/lab2/lab2_doma.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install evaluate
!pip install bert_score

In [6]:
import pandas as pd

In [7]:
data = pd.read_csv('sarcasm.txt', sep='\t')
data = data.sample(frac=1, ignore_index=True)

In [8]:
data

,Style 1,Style 2
0,just stuck my entire hand in a hot oven withou...,Self preservation isnt my strong suit
1,Being a mom with an autoimmune disease means I...,I wouldn't because sarcasm is the only weapon ...
2,Keanu reeves is killing his role in ozark seas...,The brother in ozark looks just like Keanu ree...
3,"“tit for tat, butter for fat, you kill my dog,...",My ancestors spoke about a lot of true things
4,Whoop diddy scoop poop,I would’ve just tweeted that I hated Kanye’s n...
...,...,...
688,Searching for Pokemon later because I hate mys...,I am going to play Pokémon go because I love m...
689,im glad we finally have a party in opposition ...,"I would say something like ""it is supposed to ..."
690,i love jumping on zoom meetings where everyone...,I'm sick of listening to my coworkers talk abo...
691,@goldfishbabe101 crispy all day every day over...,"""dry but honestly not mad about it."""


In [9]:
train_size = int(0.8 * len(data))
val_size = int(0.1 * len(data))

In [10]:
train_data = data[:train_size]
val_data = data[train_size:train_size + val_size]
test_data = data[train_size + val_size:]

In [11]:
train_sarcasm = train_data['Style 1'].values.tolist()
train_sincere = train_data['Style 2'].values.tolist()

In [12]:
val_sarcasm = val_data['Style 1'].values.tolist()
val_sincere = val_data['Style 2'].values.tolist()

In [13]:
test_sarcasm = test_data['Style 1'].values.tolist()
test_sincere = test_data['Style 2'].values.tolist()

In [14]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [15]:
tokenizer = AutoTokenizer.from_pretrained('t5-small')

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [16]:
inputs = tokenizer(train_sarcasm, truncation=True)
outs = tokenizer(text_target=train_sincere, truncation=True)

In [17]:
from datasets import Dataset

In [18]:
train_dataset = Dataset.from_dict({'input_ids': inputs['input_ids'],
                                   'attention_mask': inputs['attention_mask'],
                                   'labels': outs['input_ids']})

In [19]:
from transformers import DataCollatorForSeq2Seq

In [20]:
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer)

In [27]:
def train(model, train_loader, optimizer, epochs=5, device='cuda'):
  model = model.to(device)
  model.train()

  for epoch in range(epochs):
    total_loss = 0.00

    for batch in train_loader:
      batch = {k: v.to(device) for k, v in batch.items()}
      outputs = model(**batch)
      loss = outputs.loss
      optimizer.zero_grad()
      loss.backward()
      optimizer.step()
      total_loss += loss.item()

    avg_loss = total_loss / max(1, len(train_loader))
    print(f'Epoch {epoch + 1}/{epochs}, Loss: {avg_loss:.4f}')

In [28]:
from evaluate import load

In [29]:
bleu = load('bleu')
bertscore = load('bertscore')

In [33]:
def generate_and_score(model, tokenizer, prompts, targets):
  model = model.to('cuda').eval()
  predictions = []
  for prompt in prompts:
    tokens = tokenizer([prompt], return_tensors='pt').to('cuda')
    out = model.generate(**tokens, max_new_tokens=50)
    predictions.append(tokenizer.decode(out[0], skip_special_tokens=True))

  bl = bleu.compute(predictions=predictions, references=targets)['bleu']
  f1 = bertscore.compute(predictions=predictions, references=targets, model_type='distilroberta-base')['f1']

  return bl, sum(f1) / len(f1)

In [34]:
from torch.utils.data import DataLoader
from torch.optim import AdamW

best_score = 0

for lr in [0.0001, 0.0001]:
  for epochs in [2, 5]:
    for batch_size in [8, 16]:
      model = AutoModelForSeq2SeqLM.from_pretrained('t5-small')
      train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=data_collator)
      optimizer = AdamW(model.parameters(), lr=lr)
      train(model, train_loader, optimizer, epochs)

      bl, f1 = generate_and_score(model, tokenizer, val_sarcasm, val_sincere)

      print(lr, epochs, batch_size, 'BLEU: ', bl, "BERTScore: ", f1)

      if f1 > best_score:
        best_score = f1
        best_model = model

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Epoch 1/2, Loss: 3.3716
Epoch 2/2, Loss: 3.0624


config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

... (16 outputs left out)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Epoch 1/5, Loss: 3.3396
Epoch 2/5, Loss: 3.0668
Epoch 3/5, Loss: 2.8914
Epoch 4/5, Loss: 2.7716
Epoch 5/5, Loss: 2.6723
0.0001 5 8 BLEU:  0.1190722649438034 BERTScore:  0.8998319610305454


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Epoch 1/5, Loss: 3.4018
Epoch 2/5, Loss: 3.1066
Epoch 3/5, Loss: 2.9874
Epoch 4/5, Loss: 2.8799
Epoch 5/5, Loss: 2.8074
0.0001 5 16 BLEU:  0.1177010444118331 BERTScore:  0.8972972579624342


# ОБЈАСНУВАЊЕ НА РЕЗУЛТАТИТЕ
- Секоја од 8те комбинации ја евалуирав на валидациското множество со BLEU и BERTScore. Како најдобра комбинација се покажа lr=0.0001, 5 epochs и batch 16, со BERTScore 0.8973 и BLEU 0.1177.
- Изборот го направив според резултатите од BERTScore-от, бидејќи неутралната реченица користи многу различен стил и зборови од саркастичната, па затоа BLEU што е метрика која бара точни совпаѓања има ниски вредности дури и кога значењето на речениците се совпаѓа.

In [35]:
generate_and_score(best_model, tokenizer, test_sarcasm, test_sincere)

(0.0901214309017329, 0.8961809039115906)

In [36]:
model_name = 't5-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  892MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [37]:
zero_shot = []
for sample in test_sarcasm:
    zero_shot.append(f'Rewrite the following sarcastic text as a neutral text: {sample}')

In [38]:
examples = f'Input: {train_sarcasm[0]} Output: {train_sincere[0]}\nInput: {train_sarcasm[1]} Output: {train_sincere[1]}\n'

few_shot = []
for sample in test_sarcasm:
    few_shot.append(f'Examples:\n{examples}Rewrite the following sarcastic text as a neutral text: {sample} Output: ')

In [39]:
generate_and_score(model, tokenizer, zero_shot, test_sincere)

(0.008056330718020373, 0.8355640547616141)

In [40]:
generate_and_score(model, tokenizer, few_shot, test_sincere)

(0.0, 0.8318410847868238)

# ОБЈАСНУВАЊЕ ЗА РЕЗУЛТАТИТЕ
- Најдобар се покажа T5-small моделот што го трениравме (BLEU = 0.0901, BERTScore 0.8962) бидејќи научил точно како саркастичната реченица да ја претвори во неутрална. T5-base без тренирање е многу полош и со zero-shot и со few-shot, бидејќи T5 не е научен да следи инструкции и не може да не разбере што бараме од него. Кај few shot не беа од помош примерите бидејќи тој не знае како да ги искористи.